In [1]:
import os
import torch
import numpy as np
import pandas as pd
import glob
from sklearn.neighbors import KDTree


In [2]:
from src.data.data import SpacioTemporalDataset

In [3]:
file_list = [
    "s_001.csv"
]
std = SpacioTemporalDataset(
    root_dir="./data/processed/cog-load",
    file_list=file_list,
    recursive=True,
    window_length=10,
    kt=2,
    ks=2,
)

Window slice(np.int64(68510), np.int64(68511), None) too small for kt=2 and ks=2. Skipping... 
Loaded 192 graphs from ./data/processed/cog-load


In [4]:
import sys

memory_usage = sys.getsizeof(std.graphs)
print(f"Memory usage of 'std': {memory_usage} bytes ({memory_usage / 1024 / 1024:.2f} MB)")

Memory usage of 'std': 1656 bytes (0.00 MB)


In [5]:
std.graphs

[HeteroData(
   node={
     x=[504, 5],
     num_nodes=504,
   },
   (node, temporal, node)={ edge_index=[2, 2010] },
   (node, spatial, node)={ edge_index=[2, 1432] }
 ),
 HeteroData(
   node={
     x=[300, 5],
     num_nodes=300,
   },
   (node, temporal, node)={ edge_index=[2, 1194] },
   (node, spatial, node)={ edge_index=[2, 836] }
 ),
 HeteroData(
   node={
     x=[484, 5],
     num_nodes=484,
   },
   (node, temporal, node)={ edge_index=[2, 1930] },
   (node, spatial, node)={ edge_index=[2, 1312] }
 ),
 HeteroData(
   node={
     x=[243, 5],
     num_nodes=243,
   },
   (node, temporal, node)={ edge_index=[2, 966] },
   (node, spatial, node)={ edge_index=[2, 654] }
 ),
 HeteroData(
   node={
     x=[245, 5],
     num_nodes=245,
   },
   (node, temporal, node)={ edge_index=[2, 974] },
   (node, spatial, node)={ edge_index=[2, 696] }
 ),
 HeteroData(
   node={
     x=[183, 5],
     num_nodes=183,
   },
   (node, temporal, node)={ edge_index=[2, 726] },
   (node, spatial, node)={ e

In [6]:
g = std.graphs[3]

import matplotlib.pyplot as plt
import networkx as nx
from matplotlib.patches import FancyArrowPatch
%matplotlib qt
# Extract first 100 nodes
num_nodes = min(300, g["node"].num_nodes)
X_subset = g["node"].x[:num_nodes]

# Filter edges to only include those within first 100 nodes
temporal_edges = g["node", "temporal", "node"].edge_index
spatial_edges = g["node", "spatial", "node"].edge_index

# Filter temporal edges
temporal_mask = (temporal_edges[0] < num_nodes) & (temporal_edges[1] < num_nodes)
temporal_edges_filtered = temporal_edges[:, temporal_mask]

# Filter spatial edges
spatial_mask = (spatial_edges[0] < num_nodes) & (spatial_edges[1] < num_nodes)
spatial_edges_filtered = spatial_edges[:, spatial_mask]

# Create networkx graph
G = nx.Graph()
G.add_nodes_from(range(num_nodes))

# Add edges with types
for i in range(temporal_edges_filtered.shape[1]):
    src, dst = temporal_edges_filtered[0, i].item(), temporal_edges_filtered[1, i].item()
    G.add_edge(src, dst, edge_type='temporal')

for i in range(spatial_edges_filtered.shape[1]):
    src, dst = spatial_edges_filtered[0, i].item(), spatial_edges_filtered[1, i].item()
    G.add_edge(src, dst, edge_type='spatial')

# Create figure
fig, ax = plt.subplots(figsize=(16, 16))

# Use x,y coordinates from node features as positions
pos = {}
for i in range(num_nodes):
    x = X_subset[i, 1].item()  # x coordinate at index 1
    y = X_subset[i, 2].item()  # y coordinate at index 2
    pos[i] = (x, y)

# Draw temporal edges (blue) with arrows from smaller to bigger index
temporal_edge_list = [(u, v) for u, v, d in G.edges(data=True) if d.get('edge_type') == 'temporal']
for u, v in temporal_edge_list:
    # Ensure arrow goes from smaller to bigger index
    start, end = (u, v) if u < v else (v, u)
    alpha = 0.2 + (start / num_nodes) * 0.7  # Alpha from 0.2 to 0.9 based on source node index
    
    # Create arrow patch
    arrow = FancyArrowPatch(pos[start], pos[end],
                           arrowstyle='->', mutation_scale=10, 
                           color='blue', alpha=alpha, linewidth=0.5, zorder=1)
    ax.add_patch(arrow)

# Draw spatial edges (red)
spatial_edge_list = [(u, v) for u, v, d in G.edges(data=True) if d.get('edge_type') == 'spatial']
for u, v in spatial_edge_list:
    alpha = 0.2 + (u / num_nodes) * 0.7  # Alpha from 0.2 to 0.9 based on source node index
    ax.plot([pos[u][0], pos[v][0]], [pos[u][1], pos[v][1]], 
            'r-', alpha=alpha, linewidth=0.5, zorder=1)

# Draw nodes (green) with varying alpha
for node in range(num_nodes):
    alpha = 0.2 + (node / num_nodes) * 0.7
    ax.scatter(pos[node][0], pos[node][1], c='green', s=50, alpha=alpha, zorder=2)

ax.set_title(f'Graph Visualization (First {num_nodes} Nodes)', fontsize=16)
ax.set_xlabel('X coordinate')
ax.set_ylabel('Y coordinate')
plt.legend()
plt.tight_layout()

# Display in separate window
plt.show()

print(f"Visualized {num_nodes} nodes")
print(f"kt={std.kt}, ks={std.ks}")
print(f"Temporal edges: {len(temporal_edge_list)} | Et/N: {len(temporal_edge_list)/num_nodes:.2f}")
print(f"Spatial edges:  {len(spatial_edge_list)}  | Es/N: {len(spatial_edge_list)/num_nodes:.2f}")

Visualized 243 nodes
kt=2, ks=2
Temporal edges: 326 | Et/N: 1.34
Spatial edges:  329  | Es/N: 1.35


/tmp/ipykernel_3138366/3332505788.py:74: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  plt.legend()


In [7]:
file_list = [
    "s_001.csv"
]
for k in range(1, 6):
    std_temp = SpacioTemporalDataset(
        root_dir="./data/processed/cog-load",
        file_list=file_list,
        recursive=True,
        window_length=10,
        kt=k,
        ks=k,
    )
    temporal_count = 0
    spatial_count = 0
    spatial_duplicates = 0
    num_nodes = 0
    num_graphs = len(std_temp.graphs)
    for i in range(num_graphs):
        g = std_temp.graphs[i]
        temporal_count = temporal_count + g["node", "temporal", "node"].edge_index.shape[1]/2
        spatial_count = spatial_count + g["node", "spatial", "node"].edge_index.shape[1]/2
        num_nodes = num_nodes + g["node"].num_nodes
        # find duplicate edges
        temporal_edges = g["node", "temporal", "node"].edge_index.numpy()
        spatial_edges = g["node", "spatial", "node"].edge_index.numpy()
        temporal_edge_set = set()
        spatial_edge_set = set()
        
        for j in range(spatial_edges.shape[1]):
            u = spatial_edges[0, j]
            v = spatial_edges[1, j]
            edge = (u,v)
            if edge in spatial_edge_set:
                spatial_duplicates += 1
            else:
                spatial_edge_set.add(edge)
    print(f"\nkt={std_temp.kt}, ks={std_temp.ks}")
    print(f"    N={num_nodes/num_graphs} nodes per graph on average")
    print(f"    Temporal edges: {temporal_count/num_graphs} | Et/N: {temporal_count/num_nodes:.2f}")
    print(f"    Spatial edges:  {spatial_count/num_graphs}  | Es/N: {(spatial_count-spatial_duplicates)/num_nodes:.2f}")
    print(f"    Spatial duplicate edges: {spatial_duplicates/num_graphs}")

Loaded 193 graphs from ./data/processed/cog-load

kt=1, ks=1
    N=488.18134715025906 nodes per graph on average
    Temporal edges: 487.18134715025906 | Et/N: 1.00
    Spatial edges:  368.1968911917098  | Es/N: 0.75
    Spatial duplicate edges: 0.0
Window slice(np.int64(68510), np.int64(68511), None) too small for kt=2 and ks=2. Skipping... 
Loaded 192 graphs from ./data/processed/cog-load

kt=2, ks=2
    N=490.7135416666667 nodes per graph on average
    Temporal edges: 978.4270833333334 | Et/N: 1.99
    Spatial edges:  692.4713541666666  | Es/N: 1.41
    Spatial duplicate edges: 0.0
Window slice(np.int64(68510), np.int64(68511), None) too small for kt=3 and ks=3. Skipping... 
Loaded 192 graphs from ./data/processed/cog-load

kt=3, ks=3
    N=490.7135416666667 nodes per graph on average
    Temporal edges: 1466.140625 | Et/N: 2.99
    Spatial edges:  1008.8151041666666  | Es/N: 2.06
    Spatial duplicate edges: 0.0
Window slice(np.int64(68510), np.int64(68511), None) too small for kt